# Telegram Data Exploration and Network Graph
<!-- structured-notebook -->
## Notebook Summary
Purpose: characterize the collected Telegram corpus temporally and structurally before topic modeling.

Main steps:
- Load `messages.json` and filter to 2010+.
- Plot weekly and annual message volume.
- Visualize per-channel message distribution.
- Render the channel-forwarding NetworkX graph with a spring layout.

### Graph statistics

- 240 nodes (channels), 273 directed edges (forwarding links)
- Top forwarding sources: `ParasitesTheGreatAwakening` (139), `LiveHealthy` (67), `childrenshd` (29)
- Community clusters: health/longevity, alternative medicine, wellness/biohacking, vaccine-related


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Telegram/output/messages.json` | Produced by `01_data_collection/01_channel_crawl.ipynb` |
| Input | `<DATA_ROOT>/Telegram/output/tg_channel_network.gexf` | Produced by `01_data_collection/01_channel_crawl.ipynb` |
| Output | Figures rendered in-notebook | — |


## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.shared_reddit_telegram.config import PROJECT_ROOT, TELEGRAM_PIPELINE, TELEGRAM_DATA

In [ ]:
import json
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import networkx as nx

plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "figure.dpi": 120,
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.3,
    }
)

## 2. Load Data

In [ ]:
messages_path = TELEGRAM_PIPELINE / "hybrid" / "messages.json"

with open(messages_path) as f:
    data = json.load(f)

df = pd.DataFrame(data)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df = df[df["date"].dt.year >= 2010]

print(f"Total messages (2010+): {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique channels: {df['source'].nunique()}")
df.head()

## 3. Temporal Analysis

Weekly and annual message volume shows collection coverage and temporal patterns in the corpus. Spikes may correspond to major health events, supplement trends, or longevity research announcements.

In [ ]:
# Weekly message volume
weekly = df.set_index("date").resample("W").size()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly.index, weekly.values, linewidth=0.8, color="#2196F3")
ax.fill_between(weekly.index, weekly.values, alpha=0.15, color="#2196F3")
ax.set_title("Weekly Message Volume", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Messages per Week")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Annual message count
annual = df.groupby(df["date"].dt.year).size()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(annual.index.astype(int), annual.values, color="#4CAF50", edgecolor="white")
ax.set_title("Annual Message Count", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Messages")
ax.set_xticks(annual.index.astype(int))

# Add value labels on bars
for bar, val in zip(bars, annual.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(annual.values) * 0.01,
        f"{val:,}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

## 4. Channel Distribution

Message distribution across channels reveals which sources dominate the corpus. A highly skewed distribution is expected since some channels post far more frequently than others.

In [ ]:
channel_counts = Counter(df["source"])
top_channels = channel_counts.most_common(25)

channels, counts = zip(*top_channels)

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(range(len(channels)), counts, color="#FF9800", edgecolor="white")
ax.set_yticks(range(len(channels)))
ax.set_yticklabels(channels, fontsize=9)
ax.invert_yaxis()
ax.set_title("Top 25 Channels by Message Count", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Messages")

# Add count labels
for bar, val in zip(bars, counts):
    ax.text(
        bar.get_width() + max(counts) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Distribution summary
counts_series = pd.Series(list(channel_counts.values()))
print("Messages per channel (summary statistics):")
print(counts_series.describe().to_string())
print(f"\nTotal unique channels: {len(channel_counts)}")
print(f"Channels with 100+ messages: {(counts_series >= 100).sum()}")
print(f"Channels with 1000+ messages: {(counts_series >= 1000).sum()}")

## 5. Network Graph Visualization

The forwarding graph shows how information flows between Telegram health/longevity channels. Edges represent forwarding relationships: if channel A forwards a message from channel B, there is a directed edge A -> B. Highly connected nodes are influential content sources.

In [ ]:
gexf_path = TELEGRAM_PIPELINE / "hybrid" / "tg_channel_network.gexf"

G = nx.read_gexf(str(gexf_path))
print(f"Network: {G.number_of_nodes()} channels, {G.number_of_edges()} forwarding edges")

In [ ]:
pos = nx.spring_layout(G, seed=42, k=1.5 / (G.number_of_nodes() ** 0.5))

# Scale node sizes by in-degree
in_degrees = dict(G.in_degree())
node_sizes = [max(20, in_degrees.get(n, 0) * 15) for n in G.nodes]

fig, ax = plt.subplots(figsize=(14, 14))
nx.draw_networkx_edges(G, pos, alpha=0.15, arrows=True, arrowsize=6, ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color="#2196F3", alpha=0.7, ax=ax)

# Label only high-degree nodes
top_nodes = {n for n, d in in_degrees.items() if d >= 3}
labels = {n: n for n in top_nodes}
nx.draw_networkx_labels(G, pos, labels, font_size=7, font_weight="bold", ax=ax)

ax.set_title(
    f"Telegram Channel Forwarding Network ({G.number_of_nodes()} channels, {G.number_of_edges()} edges)",
    fontsize=14,
    fontweight="bold",
)
ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
print("=" * 50)
print("TELEGRAM CORPUS SUMMARY")
print("=" * 50)
print(f"Total messages:        {len(df):,}")
print(f"Unique channels:       {df['source'].nunique()}")
print(f"Date range:            {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Messages with text:    {df['text'].notna().sum():,}")
print(f"Forwarded messages:    {df['forwarded_from'].notna().sum():,}")
print()
print("Top 10 channels by volume:")
print("-" * 40)
for i, (ch, count) in enumerate(channel_counts.most_common(10), 1):
    pct = count / len(df) * 100
    print(f"  {i:>2}. {ch:<30s} {count:>6,} ({pct:.1f}%)")

In [ ]:
# Network summary
if G.number_of_nodes() > 0:
    print("\nNETWORK SUMMARY")
    print("-" * 40)
    print(f"Nodes (channels):      {G.number_of_nodes()}")
    print(f"Edges (forwards):      {G.number_of_edges()}")
    print(f"Density:               {nx.density(G):.4f}")
    if nx.is_weakly_connected(G):
        print(f"Weakly connected:      Yes")
    else:
        components = list(nx.weakly_connected_components(G))
        print(f"Weakly connected components: {len(components)}")
        print(f"Largest component size:      {len(max(components, key=len))}")

---
<!-- nav-footer -->

← [01 channel crawl](../../../../SocialMedia/Telegram/notebooks/01_data_collection/01_channel_crawl.ipynb) | [Project Overview](../../../../PROJECT_OVERVIEW.ipynb) | [01 topic modeling](../../../../SocialMedia/Telegram/notebooks/03_topic_modeling/01_topic_modeling.ipynb) →
